# LiteIE + Zero-DCE — LOLv2 Real trên Kaggle
Bật **GPU** và **Internet**, Add Input LOLv2. Chỉ sửa `DATASET_ROOT` rồi Run All.
Training/loss Zero-DCE theo hướng dẫn tạm của tác giả LiteIE; không phải loss LiteIE mới trong paper.
Train chỉ dùng toàn bộ Train/Low. Test/Normal chỉ dùng đánh giá checkpoint cuối.


In [ ]:
DATASET_ROOT = "/kaggle/input/your-lolv2-dataset"  # Chỉ sửa dòng này


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

REPO_URL = "https://github.com/mluu59990-collab/LiteIE.git"
REPO_DIR = Path("/kaggle/working/LiteIE")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, f"Unexpected repository: {origin}"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
assert (REPO_DIR / "losses.py").exists(), "Checkout chưa có training Zero-DCE"
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())
assert torch.cuda.is_available(), "Hãy bật GPU trong Settings của notebook"
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-train.txt")], check=True)


In [ ]:
# Kiểm tra cấu trúc, khả năng đọc ảnh và ghép cặp trước khi train.
command = [sys.executable, "-u", str(REPO_DIR / "train.py"),
           "--dataset-root", DATASET_ROOT, "--variant", "real"]
subprocess.run(command + ["--check-data"], cwd=REPO_DIR, check=True)


In [ ]:
# Zero-DCE defaults: 200 epoch, batch 8, resize 256, Adam lr=1e-4,
# weight decay=1e-4, gradient clip=0.1, không crop/flip.
# Tự resume run cùng recipe. Không dùng last.pt từ bản L1 cũ.
OUTPUT_DIR = Path("/kaggle/working/liteie-zero-dce-output")
RESUME = OUTPUT_DIR / "last.pt"
# Sau khi đổi session, Add Input output cũ rồi đặt RESUME tới last.pt đó.
train_command = command + ["--output-dir", str(OUTPUT_DIR)]
if RESUME.exists():
    train_command += ["--resume", str(RESUME)]
subprocess.run(train_command, cwd=REPO_DIR, check=True)


In [ ]:
import json
import shutil
metrics = OUTPUT_DIR / "test_metrics.json"
if metrics.exists():
    print(json.dumps(json.loads(metrics.read_text()), indent=2))
print("Artifacts:", [p.name for p in OUTPUT_DIR.iterdir()])
shutil.make_archive("/kaggle/working/liteie-results", "zip", OUTPUT_DIR)
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/liteie-results.zip"))


## Kết quả
- `final.pth`: weights ở epoch cuối, tương thích `model.LiteIE` và inference gốc.
- `last.pt`: model, optimizer, epoch, recipe để resume. Checkpoint L1 cũ không tương thích.
- `config.json`, `splits.json`, `history.json`, `test_metrics.json` ghi cấu hình, ảnh dùng train và kết quả.
- Train dùng toàn bộ Train/Low, không giữ validation và không đọc Train/Normal.
- Test chỉ đánh giá checkpoint cuối; RGB PSNR trung bình theo ảnh, không chỉnh sáng bằng ground truth.
Lưu output trước khi session kết thúc. Chi tiết nguồn và các điều chỉnh: `TRAINING_SOURCES.md`.
